# 📚 Merging & Concatenation
### 병합 & 연결

> **Section 8 of 11** · Pandas Complete Reference Guide for a JS/TS developer transitioning into BA  
> 전체 11개 섹션 중 **8번째** · JS/TS 개발자 출신 BA를 위한 Pandas 완전 참조 가이드

---
# 🎯 Learning Objective
Today I want to learn: / 오늘 배울 내용:
- [x] How `merge()` mirrors SQL JOIN — `inner` / `left` / `right` / `outer`, `left_on` / `right_on`, and `suffixes` for overlapping columns  
`merge()`가 SQL JOIN을 어떻게 반영하는지 — `inner` / `left` / `right` / `outer`, `left_on` / `right_on`, 겹치는 열을 위한 `suffixes`
- [x] What to check immediately after any merge to catch a silently broken join  
merge 직후 조용히 실패한 조인을 잡기 위해 무엇을 확인해야 하는지
- [x] How `concat()` stacks tables vertically or horizontally, and when `join()` is a useful shortcut  
`concat()`이 테이블을 세로 또는 가로로 쌓는 방법과, `join()`이 유용한 단축법이 되는 경우

---
# 🧠 Concept

## What is it?
*(Explain it in your own words.)*

**English**
Merging and concatenation are how you combine two or more separate tables into one. `merge()` matches rows by a shared key — like a SQL `JOIN`, or a JS array of objects matched by ID — while `concat()` simply stacks tables on top of each other, or side by side, with no key-matching involved.

**한글**
병합과 연결은 둘 이상의 별도 테이블을 하나로 합치는 방법입니다. `merge()`는 공유하는 키로 행을 매칭합니다 — SQL의 `JOIN`이나, ID로 매칭되는 JS 객체 배열과 비슷합니다. `concat()`은 키 매칭 없이 테이블을 그냥 위아래로, 또는 좌우로 쌓습니다.

## Why do we use it?
*(When is it useful?)*

**English**
Real data almost never lives in one table — order details in one file, customer info in another, one CSV exported per month. Nearly every non-trivial analysis needs at least one merge or concat before the real work even starts.

**한글**
실제 데이터는 거의 한 테이블에만 있지 않습니다 — 주문 상세는 한 파일에, 고객 정보는 다른 파일에, 매달 CSV가 하나씩 내보내집니다. 사소하지 않은 거의 모든 분석은 진짜 작업이 시작되기도 전에 최소 한 번의 merge나 concat이 필요합니다.

## When is it used in Business Analytics?
*(Real-world use case)*

**English**
"Which region does this customer belong to" (merge), "combine Q1 through Q4 into one yearly table" (concat) — these two operations show up in almost every recurring report, and getting them subtly wrong (an unintended row multiplication, a silently dropped customer) is one of the most common sources of a wrong final number.

**한글**
"이 고객은 어느 지역 소속인가"(merge), "1분기부터 4분기를 하나의 연간 테이블로 합치기"(concat) — 이 두 작업은 거의 모든 반복 보고서에 등장하며, 이를 미묘하게 잘못 다루는 것(의도치 않은 행 증식, 조용히 누락된 고객)은 최종 숫자가 틀리는 가장 흔한 원인 중 하나입니다.

### Quick Comparison: SQL vs pandas / 빠른 비교

| Concept / 개념 | SQL | pandas |
|---|---|---|
| Match rows by a key / 키로 행 매칭 | `INNER JOIN` | `pd.merge(a, b, on="key")` (default `how="inner"`) |
| Keep all left rows / 왼쪽 행 전부 유지 | `LEFT JOIN` | `pd.merge(a, b, on="key", how="left")` |
| Keep all rows from both / 양쪽 행 전부 유지 | `FULL OUTER JOIN` | `pd.merge(a, b, on="key", how="outer")` |
| Stack rows on top of each other / 행을 위아래로 쌓기 | `UNION ALL` | `pd.concat([a, b])` |
| Join on the row index instead of a column / 열 대신 행 인덱스로 조인 | (less common in SQL / SQL에서는 덜 흔함) | `a.join(b)` |

---
# 📝 Syntax

## Basic Syntax

In [1]:
import pandas as pd

orders = pd.DataFrame({
    "order_id": [1, 2, 3],
    "customer_id": [201, 202, 203],
    "amount": [45000, 32000, 61000],
})
customers = pd.DataFrame({
    "customer_id": [201, 202],
    "name": ["Minsu", "Younghee"],
})

# merge -- match rows by a shared key, like a SQL JOIN
# merge -- SQL JOIN처럼 공유하는 키로 행을 매칭
print(pd.merge(orders, customers, on="customer_id", how="left"))

   order_id  customer_id  amount      name
0         1          201   45000     Minsu
1         2          202   32000  Younghee
2         3          203   61000       NaN


## Common Variations

In [9]:
import pandas as pd

q1 = pd.DataFrame({"region": ["Seoul", "Busan"], "sales": [120, 80]})
q2 = pd.DataFrame({"region": ["Seoul", "Busan"], "sales": [150, 95]})

# concat -- stack tables on top of each other / concat -- 테이블을 위아래로 쌓기
print(pd.concat([q1, q2], ignore_index=True))
print()

# inner join (default) -- only rows present in BOTH tables / inner join(기본값) -- 양쪽 모두에 있는 행만
orders = pd.DataFrame({"customer_id": [201, 202, 999], "amount": [45000, 32000, 10000]})
customers = pd.DataFrame({"customer_id": [201, 202], "name": ["Minsu", "Younghee"]})
print(pd.merge(orders, customers, on="customer_id"))

  region  sales
0  Seoul    120
1  Busan     80
2  Seoul    150
3  Busan     95

   customer_id  amount      name
0          201   45000     Minsu
1          202   32000  Younghee


---
# 🧪 Small Examples

## Example 1 — merge: SQL JOIN Equivalents
*(Covers source section 8-1)*

**English:** `how="inner"` (the default) keeps only rows matched on both sides; `"left"` keeps every row from the left table; `"outer"` keeps every row from both. `left_on` / `right_on` handle key columns with different names, and `suffixes` disambiguate columns that exist in both tables. Watch for **1:N row expansion**: merging onto a table where the key repeats duplicates the other side's data across every match.  
**한글:** `how="inner"`(기본값)은 양쪽에서 매칭된 행만 유지하고, `"left"`는 왼쪽 테이블의 모든 행을 유지하며, `"outer"`는 양쪽의 모든 행을 유지합니다. `left_on` / `right_on`은 이름이 다른 키 열을 처리하고, `suffixes`는 양쪽 테이블에 모두 있는 열을 구분합니다. **1:N 행 증식**을 주의하세요: 키가 반복되는 테이블에 merge하면 다른 쪽의 데이터가 매칭될 때마다 중복됩니다.

In [3]:
import pandas as pd

orders = pd.DataFrame({
    "order_id": [1001, 1002, 1003, 1004, 1005],
    "customer_id": [201, 202, 203, 201, 204],   # 204 has no matching customer / 204는 매칭되는 고객이 없음
    "amount": [45000, 32000, 45000, 61000, 28000],
})
customers = pd.DataFrame({
    "customer_id": [201, 202, 203, 205],        # 205 has no matching order / 205는 매칭되는 주문이 없음
    "name": ["Minsu", "Younghee", "Junho", "Seoyeon"],
})

print("how='inner' -- only matched rows (204 and 205 both dropped):")
print(pd.merge(orders, customers, on="customer_id", how="inner"))
print()

print("how='left' -- every order kept, unmatched customer info becomes NaN:")
print(pd.merge(orders, customers, on="customer_id", how="left"))
print()

print("how='outer' -- everything from both sides kept:")
print(pd.merge(orders, customers, on="customer_id", how="outer"))
print()

# left_on / right_on -- key columns with different names / 이름이 다른 키 열
o2 = pd.DataFrame({"order_id": [1, 2], "cust_id": [201, 202], "amount": [45000, 32000]})
c2 = pd.DataFrame({"id": [201, 202], "name": ["Minsu", "Younghee"]})
print("left_on / right_on:")
print(pd.merge(o2, c2, left_on="cust_id", right_on="id", how="left"))
print()

# suffixes -- disambiguate overlapping column names / 겹치는 열 이름 구분
d2023 = pd.DataFrame({"id": [1, 2], "score": [85, 90]})
d2024 = pd.DataFrame({"id": [1, 2], "score": [80, 95]})
print("suffixes:")
print(pd.merge(d2023, d2024, on="id", suffixes=("_2023", "_2024")))
print()

# 1:N expansion -- watch the row count / 1:N 증식 -- 행 개수를 주시할 것
small_customers = pd.DataFrame({"cid": [1, 2], "name": ["Alice", "Bob"]})
many_orders = pd.DataFrame({"oid": [101, 102, 103, 104], "cid": [1, 1, 2, 1]})
print(f"customers: {len(small_customers)} rows, orders: {len(many_orders)} rows")
expanded = pd.merge(small_customers, many_orders, on="cid")
print(f"merged: {len(expanded)} rows -- Alice appears 3 times, once per matching order")
print(expanded)

how='inner' -- only matched rows (204 and 205 both dropped):
   order_id  customer_id  amount      name
0      1001          201   45000     Minsu
1      1002          202   32000  Younghee
2      1003          203   45000     Junho
3      1004          201   61000     Minsu

how='left' -- every order kept, unmatched customer info becomes NaN:
   order_id  customer_id  amount      name
0      1001          201   45000     Minsu
1      1002          202   32000  Younghee
2      1003          203   45000     Junho
3      1004          201   61000     Minsu
4      1005          204   28000       NaN

how='outer' -- everything from both sides kept:
   order_id  customer_id   amount      name
0    1001.0          201  45000.0     Minsu
1    1004.0          201  61000.0     Minsu
2    1002.0          202  32000.0  Younghee
3    1003.0          203  45000.0     Junho
4    1005.0          204  28000.0       NaN
5       NaN          205      NaN   Seoyeon

left_on / right_on:
   order_id  cust_

## Example 2 — Post-Merge Quality Checks
*(Covers source section 8-2)*

**English:** A merge can fail silently — rows just quietly don't match, and nothing raises an error. Three checks, run immediately after every merge, catch this: compare `len()` before and after, check `.isna().sum()` for unexpected new missing values, and look directly at the unmatched rows to see *why* they failed.  
**한글:** merge는 조용히 실패할 수 있습니다 — 행이 그냥 조용히 매칭되지 않을 뿐, 아무 오류도 나지 않습니다. merge 직후 실행하는 세 가지 확인이 이를 잡아냅니다: merge 전후의 `len()`을 비교하고, 예상치 못한 새 결측치를 `.isna().sum()`으로 확인하고, 매칭되지 않은 행을 직접 살펴서 *왜* 실패했는지 봅니다.

In [4]:
import pandas as pd

orders = pd.DataFrame({
    "order_id": [1001, 1002, 1003, 1004, 1005],
    "customer_id": [201, 202, 203, 201, 204],
    "amount": [45000, 32000, 45000, 61000, 28000],
})
customers = pd.DataFrame({
    "customer_id": [201, 202, 203, 205],
    "name": ["Minsu", "Younghee", "Junho", "Seoyeon"],
    "region": ["Seoul", "Busan", "Seoul", "Incheon"],
})

merged = pd.merge(orders, customers, on="customer_id", how="left")

# Check 1 -- did the row count change unexpectedly? / 확인 1 -- 행 개수가 예상치 못하게 바뀌었는가?
print(f"left: {len(orders)} rows, right: {len(customers)} rows, merged: {len(merged)} rows")
print()

# Check 2 -- any unexpected missing values? / 확인 2 -- 예상치 못한 결측치가 있는가?
print("missing values (possible join failures):")
print(merged.isna().sum())
print()

# Check 3 -- look directly at what failed to match / 확인 3 -- 매칭 실패한 행을 직접 확인
print("unmatched rows:")
print(merged[merged["name"].isna()])

left: 5 rows, right: 4 rows, merged: 5 rows

missing values (possible join failures):
order_id       0
customer_id    0
amount         0
name           1
region         1
dtype: int64

unmatched rows:
   order_id  customer_id  amount name region
4      1005          204   28000  NaN    NaN


## Example 3 — concat: Stacking & Side-by-Side
*(Covers source section 8-3)*

**English:** `pd.concat([a, b])` with the default `axis=0` stacks rows on top of each other — if the two tables' columns don't fully match, the missing ones fill with `NaN` rather than raising an error. `axis=1` glues tables side by side, aligned by the **row index** (not by any key). `keys=[...]` tags each source table so you can tell where a stacked row originally came from.  
**한글:** 기본값 `axis=0`을 쓰는 `pd.concat([a, b])`는 행을 위아래로 쌓습니다 — 두 테이블의 열이 완전히 일치하지 않으면, 없는 열은 오류 대신 `NaN`으로 채워집니다. `axis=1`은 (키가 아니라) **행 인덱스**로 정렬해서 테이블을 좌우로 붙입니다. `keys=[...]`는 각 원본 테이블에 태그를 붙여서, 쌓인 행이 원래 어디서 왔는지 알 수 있게 합니다.

In [11]:
import pandas as pd

# axis=0 (default) -- stack rows / axis=0(기본값) -- 행을 쌓기
q1 = pd.DataFrame({"region": ["Seoul", "Busan"], "sales": [120, 80], "quarter": ["Q1", "Q1"]})
q2 = pd.DataFrame({"region": ["Seoul", "Busan"], "sales": [150, 95], "quarter": ["Q2", "Q2"]})
print("concat, axis=0:")
print(pd.concat([q1, q2], ignore_index=True))
print()

# Mismatched columns -- missing ones fill with NaN, no error / 열이 다르면 -- 오류 대신 NaN으로 채워짐
df_a = pd.DataFrame({"a": [1, 2], "b": [3, 4]})
df_b = pd.DataFrame({"b": [5, 6], "c": [7, 8]})
print("concat with mismatched columns:")
print(pd.concat([df_a, df_b], ignore_index=True))
print()

# axis=1 -- side by side, aligned by row index / axis=1 -- 행 인덱스로 정렬해서 좌우로 붙임
info = pd.DataFrame({"region": ["Seoul", "Busan"], "population": [9700000, 3400000]})
sales_data = pd.DataFrame({"sales": [150, 95], "cost": [100, 65]})
print("concat, axis=1:")
print(pd.concat([info, sales_data], axis=1))
print()

# keys= -- track which table each row came from / keys= -- 각 행의 원본 테이블 추적
tagged = pd.concat([q1, q2], keys=["Q1", "Q2"])
print("concat with keys=:")
print(tagged)

concat, axis=0:
  region  sales quarter
0  Seoul    120      Q1
1  Busan     80      Q1
2  Seoul    150      Q2
3  Busan     95      Q2

concat with mismatched columns:
     a  b    c
0  1.0  3  NaN
1  2.0  4  NaN
2  NaN  5  7.0
3  NaN  6  8.0

concat, axis=1:
  region  population  sales  cost
0  Seoul     9700000    150   100
1  Busan     3400000     95    65

concat with keys=:
     region  sales quarter
Q1 0  Seoul    120      Q1
   1  Busan     80      Q1
Q2 0  Seoul    150      Q2
   1  Busan     95      Q2


## Example 4 — join: Index-Based Merge Shortcut
*(Covers source section 8-4)*

**English:** `left.join(right)` is a shortcut for a left merge that uses the **row index** as the join key instead of a column — convenient once both tables already share a meaningful index (typically after `set_index()` on the same ID). For anything more flexible than that, plain `merge()` is usually clearer.  
**한글:** `left.join(right)`는 열 대신 **행 인덱스**를 조인 키로 사용하는 left merge의 단축형입니다 — 두 테이블이 이미 같은 의미 있는 인덱스를 공유할 때(보통 같은 ID로 `set_index()`한 뒤) 편리합니다. 그 이상의 유연함이 필요하다면 보통 일반 `merge()`가 더 명확합니다.

In [6]:
import pandas as pd

left = pd.DataFrame({"sales": [100, 200, 150]}, index=["Seoul", "Busan", "Incheon"])
right = pd.DataFrame({"population": [9700000, 3400000, 3000000]}, index=["Seoul", "Busan", "Incheon"])

print("join() -- merges on the shared index, no 'on=' needed:")
print(left.join(right))

join() -- merges on the shared index, no 'on=' needed:
         sales  population
Seoul      100     9700000
Busan      200     3400000
Incheon    150     3000000


## Example 5 (Practice) — Fill in the Blanks
*(Practice built from the merge + quality-check + concat combo -- the source PDF has no separate numbered practice problem for this section)*

**English:** Fill in each `________` blank below. The code is syntactically valid Python, so it won't raise a `SyntaxError` — but it also won't print any result until every blank is correct (it will raise a runtime error instead, which is expected).
**한글:** 아래 `________` 빈칸을 채워보세요. 코드는 문법적으로 올바른 파이썬이라 `SyntaxError`는 나지 않지만, 모든 빈칸이 정확해지기 전까지는 결과가 출력되지 않습니다(대신 런타임 오류가 나는데, 이는 의도된 동작입니다).

In [12]:
import pandas as pd

products = pd.DataFrame({
    "product_id": ["P01", "P02", "P03", "P04"],
    "product_name": ["Laptop", "Mouse", "Keyboard", "Monitor"],
})
sales_q1 = pd.DataFrame({
    "product_id": ["P01", "P02", "P03", "P05"],   # P05 doesn't exist in products / P05는 products에 없음
    "units_sold": [10, 25, 15, 8],
})
sales_q2 = pd.DataFrame({
    "product_id": ["P01", "P02"],
    "units_sold": [12, 30],
})

# 1. Left-merge sales_q1 with products on "product_id"
#    sales_q1과 products를 "product_id" 기준으로 left merge
merged = pd.merge(sales_q1, products, on="product_id", how="left")

# 2. Count how many rows failed to match (missing product_name)
#    매칭 실패한 행 개수 세기 (product_name이 결측인 행)
unmatched = merged["product_name"].isna().sum()

# 3. Stack Q1 and Q2 sales into one long table
#    Q1과 Q2 매출을 하나의 긴 테이블로 쌓기
combined = pd.concat([sales_q1, sales_q2], ignore_index=True)

print(merged)
print("unmatched rows:", unmatched)
print(combined)

  product_id  units_sold product_name
0        P01          10       Laptop
1        P02          25        Mouse
2        P03          15     Keyboard
3        P05           8          NaN
unmatched rows: 1
  product_id  units_sold
0        P01          10
1        P02          25
2        P03          15
3        P05           8
4        P01          12
5        P02          30


### 💡 Hint / 힌트
`merge` · `"left"` · `isna` · `concat`

### ✅ Solution / 정답
*(Try solving it yourself first! / 먼저 스스로 풀어본 뒤에 확인하세요!)*

In [ ]:
import pandas as pd

products = pd.DataFrame({
    "product_id": ["P01", "P02", "P03", "P04"],
    "product_name": ["Laptop", "Mouse", "Keyboard", "Monitor"],
})
sales_q1 = pd.DataFrame({
    "product_id": ["P01", "P02", "P03", "P05"],
    "units_sold": [10, 25, 15, 8],
})
sales_q2 = pd.DataFrame({
    "product_id": ["P01", "P02"],
    "units_sold": [12, 30],
})

merged = pd.merge(sales_q1, products, on="product_id", how="left")
unmatched = merged["product_name"].isna().sum()
combined = pd.concat([sales_q1, sales_q2], ignore_index=True)

print(merged)
print("unmatched rows:", unmatched)
print(combined)

# P05 has no match in products -- it's a discontinued or mistyped product code,
# exactly the kind of thing a post-merge quality check (Example 2) is meant to catch.
# P05는 products에 매칭되는 게 없음 -- 단종되었거나 오타난 상품 코드일 가능성이 높으며,
# 이런 걸 잡아내는 게 바로 merge 후 품질 확인(Example 2)의 목적입니다.

---
# ⚠️ Common Mistakes

### Mistake 1 — Not specifying `how=` and assuming nothing gets dropped
**English:** `pd.merge(orders, customers, on="customer_id")` silently defaults to `how="inner"` — any order without a matching customer (or vice versa) simply disappears from the result, with no warning at all.  
**한글:** `pd.merge(orders, customers, on="customer_id")`는 조용히 `how="inner"`를 기본값으로 사용합니다 — 매칭되는 고객이 없는 주문(또는 그 반대)은 아무 경고 없이 결과에서 그냥 사라집니다.

**✅ Fix / 해결법:**  
If every row of your main table needs to survive the merge, always write `how="left"` explicitly — never rely on the default.  
메인 테이블의 모든 행이 merge 후에도 살아남아야 한다면, 항상 `how="left"`를 명시적으로 쓰세요 — 기본값에 의존하지 마세요.

### Mistake 2 — Not checking for row-count expansion after a merge
**English:** When the key repeats on one side (a 1:N relationship, like one customer with many orders), merging duplicates the "1" side's data across every match. Skipping the `len()` check from Example 2 means an unexpected row-count jump goes unnoticed — and any `.sum()` computed afterward will be silently inflated.  
**한글:** 한쪽에서 키가 반복되면(1:N 관계, 예를 들어 고객 한 명에 주문 여러 개) merge는 "1" 쪽의 데이터를 매칭될 때마다 중복시킵니다. Example 2의 `len()` 확인을 건너뛰면 예상치 못한 행 개수 증가를 놓치게 되고 — 이후 계산하는 어떤 `.sum()`이든 조용히 부풀려집니다.

**✅ Fix / 해결법:**  
Always compare `len()` before and after a merge, and know in advance whether the relationship between the two tables is 1:1, 1:N, or N:N.  
항상 merge 전후의 `len()`을 비교하고, 두 테이블의 관계가 1:1인지, 1:N인지, N:N인지 미리 파악하세요.

### Mistake 3 — Assuming `concat()` checks that columns actually match
**English:** `pd.concat([df_a, df_b])` will happily stack two tables even if their columns mean completely different things — as long as the *names* happen to overlap, or don't, it just fills gaps with `NaN` rather than raising an error about a structural mismatch.  
**한글:** `pd.concat([df_a, df_b])`는 두 테이블의 열이 완전히 다른 것을 의미하더라도 순순히 쌓습니다 — 열 *이름*이 우연히 겹치거나 겹치지 않거나 상관없이, 구조 불일치에 대한 오류 없이 그냥 빈 칸을 `NaN`으로 채웁니다.

**✅ Fix / 해결법:**  
Before concatenating tables from different sources, manually verify the columns represent the same things — `concat()` trusts you completely and won't catch a mistake here.  
다른 출처의 테이블을 연결하기 전에, 열이 정말 같은 것을 나타내는지 직접 확인하세요 — `concat()`은 여러분을 전적으로 신뢰하며 여기서 실수를 잡아주지 않습니다.

---
# 💡 Tips
Useful tips or shortcuts / 유용한 팁과 단축법

- Immediately after any merge, run the 3-line check from Example 2: `len()`, `.isna().sum()`, and a look at the unmatched rows — catching a broken join takes seconds; debugging a wrong number derived from one can take hours.  
merge 직후에는 항상 Example 2의 3줄짜리 확인을 실행하세요: `len()`, `.isna().sum()`, 매칭 실패 행 확인 — 깨진 조인을 잡는 데는 몇 초면 되지만, 거기서 비롯된 잘못된 숫자를 디버깅하는 데는 몇 시간이 걸릴 수 있습니다.
- Default to `how="left"` when the goal is "add information to my main table without losing any of its rows" — it's the safest, most predictable option for typical reporting.  
"메인 테이블에 정보를 추가하되 행은 하나도 잃지 않기"가 목표라면 `how="left"`를 기본으로 사용하세요 — 일반적인 보고서 작성에 가장 안전하고 예측 가능한 선택지입니다.
- `concat()` never checks that columns mean the same thing across tables — that verification is on you, before combining data from different sources.  
`concat()`은 테이블 간에 열이 같은 것을 의미하는지 절대 확인하지 않습니다 — 다른 출처의 데이터를 합치기 전에 그 확인은 여러분의 몫입니다.
- `join()` is really just `merge(how="left")` using the index as the key — reach for it only when both tables already share a meaningful index (like after `set_index()` on the same ID).  
`join()`은 사실 인덱스를 키로 쓰는 `merge(how="left")`일 뿐입니다 — 두 테이블이 이미 같은 의미 있는 인덱스를 공유할 때만(같은 ID로 `set_index()`한 뒤 등) 사용하세요.

---
# 🔗 Related Concepts

```
Aggregation & GroupBy      (Section 7 -- a groupby summary often needs merging back onto the original table)
    ↓
Merging & Concatenation      ← you are here / 지금 여기 (Section 8)
    ↓
Pivot Table & Reshape        (Section 9 -- often the very next step after combining tables)
    ↓
Time Series -> BA Techniques
```

*How is today's topic connected to other concepts?*

**English:** A `groupby().agg()` summary from Section 7 shrinks to one row per group — merging it back onto the original, full-size table (matched on the same group key) is one of the most common real uses of today's `merge()`. Looking ahead, Section 9's `pivot_table()` often runs immediately after a merge or concat, once several sources have become one combined table ready to be reshaped.

**한글:** 7번 섹션의 `groupby().agg()` 요약은 그룹당 한 행으로 줄어듭니다 — 이를 (같은 그룹 키로 매칭해서) 원본의 전체 크기 테이블에 다시 붙이는 것이 오늘 배운 `merge()`의 가장 흔한 실무 활용 중 하나입니다. 앞을 내다보면, 9번 섹션의 `pivot_table()`은 여러 출처가 하나의 결합된 테이블이 되어 재구조화될 준비가 된 직후, 즉 merge나 concat 바로 다음에 자주 실행됩니다.

---
# 💼 Business Example
*How would a Business Analyst use this?*

**Scenario / 시나리오**

**English:** You have a Q1 orders export and a Q2 orders export, plus a shared customer master list. Combine both quarters into one table, attach customer names, and flag any order that couldn't be matched to a known customer.

**한글:** 1분기 주문 내보내기 파일과 2분기 주문 내보내기 파일, 그리고 공유 고객 마스터 목록이 있습니다. 두 분기를 하나의 테이블로 합치고, 고객명을 붙이고, 알려진 고객과 매칭되지 않는 주문을 표시하세요.

**To Do / 할 일**
- [x] Combine both quarters into one table with `concat()`  
`concat()`으로 두 분기를 하나의 테이블로 합치기
- [x] Attach customer names with a left `merge()`  
left `merge()`로 고객명 붙이기
- [x] Run the post-merge quality check  
merge 후 품질 확인 실행하기
- [x] List any orders with no matching customer  
매칭되는 고객이 없는 주문 나열하기

In [13]:
import pandas as pd

q1_orders = pd.DataFrame({
    "order_id": [1001, 1002, 1003],
    "customer_id": [201, 202, 203],
    "amount": [45000, 32000, 61000],
})
q2_orders = pd.DataFrame({
    "order_id": [1004, 1005, 1006],
    "customer_id": [201, 204, 202],   # 204 is not in the customer master list / 204는 고객 마스터에 없음
    "amount": [28000, 55000, 40000],
})
customers = pd.DataFrame({
    "customer_id": [201, 202, 203],
    "name": ["Minsu", "Younghee", "Junho"],
})

# 1) Combine both quarters / 두 분기 합치기
all_orders = pd.concat([q1_orders, q2_orders], ignore_index=True)

# 2) Attach customer names / 고객명 붙이기
enriched = pd.merge(all_orders, customers, on="customer_id", how="left")

# 3) Quality check / 품질 확인
print(f"total orders: {len(all_orders)}, after merge: {len(enriched)}")
unmatched = enriched[enriched["name"].isna()]
print(f"orders with no matching customer: {len(unmatched)}")
print(unmatched)
print()
print(enriched)

total orders: 6, after merge: 6
orders with no matching customer: 1
   order_id  customer_id  amount name
4      1005          204   55000  NaN

   order_id  customer_id  amount      name
0      1001          201   45000     Minsu
1      1002          202   32000  Younghee
2      1003          203   61000     Junho
3      1004          201   28000     Minsu
4      1005          204   55000       NaN
5      1006          202   40000  Younghee


---
# 📝 Summary
*Write today's concept in 3~5 sentences.*

**English**
Merging and concatenation combine separate tables into one, in two fundamentally different ways. `merge()` matches rows by a shared key, mirroring SQL's `JOIN` — `how="inner"` (the default) keeps only matched rows, `"left"` keeps every row from the main table, and `"outer"` keeps everything from both, while `left_on` / `right_on` and `suffixes` handle mismatched key names and overlapping columns. A merge can fail silently, so `len()`, `.isna().sum()`, and a direct look at unmatched rows are the standard three-step check to run right after. `concat()` simply stacks tables — vertically by default (mismatched columns become `NaN`, not an error) or side by side with `axis=1` — and `join()` is a convenient shortcut for merging on the row index instead of a column.

**한글**
병합과 연결은 근본적으로 다른 두 가지 방식으로 별도의 테이블을 하나로 합칩니다. `merge()`는 SQL의 `JOIN`을 반영해서 공유하는 키로 행을 매칭합니다 — `how="inner"`(기본값)는 매칭된 행만 유지하고, `"left"`는 메인 테이블의 모든 행을 유지하며, `"outer"`는 양쪽 전부를 유지합니다. `left_on` / `right_on`과 `suffixes`는 이름이 다른 키와 겹치는 열을 처리합니다. merge는 조용히 실패할 수 있으므로, `len()`, `.isna().sum()`, 매칭 실패 행 직접 확인이 merge 직후 실행하는 표준 3단계 확인입니다. `concat()`은 그냥 테이블을 쌓습니다 — 기본은 세로로(열이 다르면 오류 대신 `NaN`) 또는 `axis=1`로 좌우로 — 그리고 `join()`은 열 대신 행 인덱스로 merge하는 편리한 단축법입니다.

---
# 📌 One Sentence Summary
Today's topic in ONE sentence.

> Merging matches rows by a shared key like a SQL JOIN, concatenation simply stacks tables together with no matching involved, and a quick post-merge quality check is what separates a trustworthy combined table from a silently broken one.

> 병합은 SQL JOIN처럼 공유하는 키로 행을 매칭하고, 연결은 매칭 없이 그냥 테이블을 쌓으며, 빠른 merge 후 품질 확인이 믿을 수 있는 결합 테이블과 조용히 깨진 테이블을 가릅니다.

---
# ❓ Review Questions

**Q1.** What's the difference between `how="inner"` and `how="left"` in `pd.merge()`, and which one silently drops rows?  
**Q1.** `pd.merge()`에서 `how="inner"`와 `how="left"`의 차이는 무엇이며, 어느 쪽이 조용히 행을 삭제하나요?

inner keeps only matching keys from both DataFrames, while left keeps every row from the left DataFrame. inner silently drops non-matching rows.  
inner는 양쪽 DataFrame에 모두 존재하는 key만 남기고, left는 왼쪽 DataFrame의 모든 행을 유지합니다. inner가 key가 매칭되지 않는 행을 조용히 삭제.

**Q2.** What three things should you check immediately after any merge?  
**Q2.** merge 직후에는 어떤 세 가지를 확인해야 하나요?

row count, key duplication/matching behavior, unexpected NaN values.
행 개수, key의 중복/매칭 상태, 예상치 못한 NaN.

**Q3.** If a customer with 2 orders gets merged with an orders table, how many rows does that customer's info end up on, and why?  
**Q3.** 주문이 2건인 고객이 주문 테이블과 merge되면, 그 고객의 정보는 몇 개의 행에 나타나며 그 이유는 무엇인가요?

It appears on 2 rows because the customer's information is repeated for each of their 2 orders.
2개. 한 고객에게 주문이 2개 있기 때문에 고객 정보가 각 주문 행마다 반복.

**Q4.** What happens when you `pd.concat()` two DataFrames that don't have exactly the same columns?  
**Q4.** 열이 완전히 같지 않은 두 DataFrame을 `pd.concat()`하면 어떻게 되나요?

Missing columns are filled with NaN, and the result contains the union of all columns.  
없는 column은 NaN으로 채워지고, 모든 column을 합친 형태가 됨.  

**Q5.** When is `.join()` a good shortcut instead of `pd.merge()`?  
**Q5.** `pd.merge()` 대신 `.join()`이 좋은 단축법이 되는 경우는 언제인가요?

.join() is a convenient shortcut when you want to combine two DataFrames based on their indexes.  
.join()은 두 DataFrame을 index를 기준으로 간단하게 붙일 때 좋은 shortcut.

---
*📅 Try answering these again in a few days. / 며칠 뒤에 다시 답해보세요.*